# Baseline ResNet-50 thường (không distill) — 3 lớp glass / paper / plastic

So sánh với student CAKD (ResNet-50 học từ teacher ViT).

**Chuẩn bị:** Settings → Accelerator → **GPU T4/P100** → Add Input: dataset `15k-image-trash`.

Chạy lần lượt từng cell từ trên xuống. Kết quả trong `/kaggle/working/baseline`:
- `resnet50_baseline_best.pth` — trọng số tốt nhất theo val acc (**lấy file này để dùng**)
- `history_baseline.json` — lịch sử từng epoch, cùng format với `history_cakd.json` để vẽ so sánh

In [ ]:
# Ô 1 — Kiểm tra dataset đã gắn đúng chưa (phải in ra: glass  paper  plastic)
!ls /kaggle/input
!ls /kaggle/input/15k-image-trash/15K_Image

In [ ]:
# Ô 2 — Cấu hình (sửa ở đây nếu cần)
CONFIG = dict(
    data_path="/kaggle/input/15k-image-trash/15K_Image",
    classes=["glass", "paper", "plastic"],
    output_dir="/kaggle/working/baseline",
    epochs=30,
    batch_size=32,
    lr=0.01,           # SGD, đã hợp với batch 32 + pretrained
    momentum=0.9,
    weight_decay=1e-4,
    workers=2,
    img_size=224,
    pretrained=True,   # nạp trọng số ImageNet (ResNet-50 chuẩn torchvision)
)
CONFIG

In [ ]:
# Ô 3 — Dataset: đọc thẳng cấu trúc <class>/<split>/images/*.jpg (bỏ qua labels/)
import json, os, time

import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")


class TrashDataset(Dataset):
    def __init__(self, root, split, classes, transform):
        self.transform = transform
        self.samples = []
        for idx, cls in enumerate(classes):
            img_dir = os.path.join(root, cls, split, "images")
            if not os.path.isdir(img_dir):
                raise FileNotFoundError(f"Không thấy {img_dir} — kiểm tra data_path/classes ở Ô 2")
            for f in sorted(os.listdir(img_dir)):
                if os.path.splitext(f)[1].lower() in IMG_EXTS:
                    self.samples.append((os.path.join(img_dir, f), idx))
        if not self.samples:
            raise RuntimeError(f"Không có ảnh nào trong split '{split}'")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        return self.transform(Image.open(path).convert("RGB")), label


# Augment chuẩn cho finetune pretrained (mức "bình thường": không ta_wide/mixup/EMA)
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(CONFIG["img_size"]),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    normalize,
])
eval_tf = transforms.Compose([
    transforms.Resize(int(CONFIG["img_size"] * 256 / 224)),
    transforms.CenterCrop(CONFIG["img_size"]),
    transforms.ToTensor(),
    normalize,
])

loaders = {}
for split, tf, shuffle in [("train", train_tf, True), ("val", eval_tf, False), ("test", eval_tf, False)]:
    ds = TrashDataset(CONFIG["data_path"], split, CONFIG["classes"], tf)
    loaders[split] = DataLoader(ds, batch_size=CONFIG["batch_size"], shuffle=shuffle,
                                num_workers=CONFIG["workers"], pin_memory=True)
    print(f"{split:5s}: {len(ds)} ảnh")

In [ ]:
# Ô 4 — Model ResNet-50 chuẩn torchvision + loss + optimizer + lịch lr
from torchvision.models import ResNet50_Weights, resnet50

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

weights = ResNet50_Weights.IMAGENET1K_V2 if CONFIG["pretrained"] else None
model = resnet50(weights=weights)
model.fc = nn.Linear(model.fc.in_features, len(CONFIG["classes"]))  # 1000 lớp -> 3 lớp
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=CONFIG["lr"],
                            momentum=CONFIG["momentum"], weight_decay=CONFIG["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["epochs"])
scaler = torch.amp.GradScaler() if device.type == "cuda" else None
print("Model sẵn sàng — pretrained:", CONFIG["pretrained"])

In [ ]:
# Ô 5 — TRAIN (cell chạy lâu nhất). Mỗi epoch: train -> đo val -> lưu best + history
def run_epoch(model, loader, training):
    """Trả về (loss trung bình, acc@1 %). training=False -> chỉ đánh giá."""
    model.train(training)
    total_loss, correct, seen = 0.0, 0, 0
    ctx = torch.enable_grad() if training else torch.inference_mode()
    with ctx:
        for image, target in loader:
            image = image.to(device, non_blocking=True)
            target = target.to(device, non_blocking=True)
            with torch.autocast(device.type, enabled=scaler is not None):
                output = model(image)
                loss = criterion(output, target)
            if training:
                optimizer.zero_grad(set_to_none=True)
                if scaler is not None:
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    optimizer.step()
            total_loss += loss.item() * image.size(0)
            correct += (output.argmax(1) == target).sum().item()
            seen += image.size(0)
    return total_loss / seen, 100.0 * correct / seen


os.makedirs(CONFIG["output_dir"], exist_ok=True)
best_path = os.path.join(CONFIG["output_dir"], "resnet50_baseline_best.pth")
history, best_acc, best_epoch = [], 0.0, -1

for epoch in range(CONFIG["epochs"]):
    t0 = time.time()
    train_loss, train_acc = run_epoch(model, loaders["train"], training=True)
    val_loss, val_acc = run_epoch(model, loaders["val"], training=False)
    lr_now = optimizer.param_groups[0]["lr"]
    scheduler.step()

    # cùng format với history_cakd.json để tools/plot_report.py vẽ so sánh được
    history.append({
        "epoch": epoch,
        "train_loss": round(train_loss, 5),
        "train_acc1": round(train_acc, 5),
        "test_acc1": round(val_acc, 5),
        "cls_loss": None, "pca_loss": None, "gl_loss": None, "gan_loss": None,
        "lr": round(lr_now, 8),
    })
    with open(os.path.join(CONFIG["output_dir"], "history_baseline.json"), "w") as f:
        json.dump(history, f, indent=2)

    flag = ""
    if val_acc > best_acc:  # lưu best theo val acc -> không mất epoch đỉnh
        best_acc, best_epoch = val_acc, epoch
        torch.save({"model": model.state_dict(), "classes": CONFIG["classes"],
                    "epoch": epoch, "val_acc1": val_acc}, best_path)
        flag = "  <- best, đã lưu"
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(), "epoch": epoch,
                "classes": CONFIG["classes"]},
               os.path.join(CONFIG["output_dir"], "checkpoint_last.pth"))

    print(f"Epoch {epoch:3d}/{CONFIG['epochs']}  lr {lr_now:.5f}  "
          f"train loss {train_loss:.4f} acc {train_acc:.2f}%  |  val acc {val_acc:.2f}%"
          f"  ({time.time() - t0:.0f}s){flag}")

print(f"\nBest val acc@1 = {best_acc:.2f}% @ epoch {best_epoch}")

In [ ]:
# Ô 6 — Chấm bộ TỐT NHẤT trên tập test (chỉ chạy 1 lần, sau khi đã chọn xong model)
model.load_state_dict(torch.load(best_path, map_location=device)["model"])
test_loss, test_acc = run_epoch(model, loaders["test"], training=False)
print(f"TEST acc@1 (bộ tốt nhất, epoch {best_epoch}) = {test_acc:.2f}%")

In [ ]:
# Ô 7 — Vẽ nhanh quá trình train (loss / accuracy / lr)
import matplotlib.pyplot as plt

ep = [r["epoch"] for r in history]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
axes[0].plot(ep, [r["train_loss"] for r in history], color="#2a78d6", lw=2, label="train loss")
axes[0].set_title("Loss (train)"); axes[0].legend(frameon=False)
axes[1].plot(ep, [r["train_acc1"] for r in history], color="#2a78d6", lw=2, label="train acc@1")
axes[1].plot(ep, [r["test_acc1"] for r in history], color="#eb6834", lw=2, label="val acc@1")
axes[1].plot([best_epoch], [best_acc], "o", ms=9, color="#eb6834", mec="white", mew=2)
axes[1].annotate(f"best {best_acc:.2f}% @ep{best_epoch}", (best_epoch, best_acc),
                 xytext=(0, 12), textcoords="offset points", ha="center", fontweight="bold", fontsize=9)
axes[1].margins(y=0.18); axes[1].set_title("Accuracy"); axes[1].legend(frameon=False, loc="lower right")
axes[2].plot(ep, [r["lr"] for r in history], color="#1baf7a", lw=2, label="lr")
axes[2].set_title("Learning rate"); axes[2].legend(frameon=False)
for ax in axes:
    ax.grid(alpha=0.3); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.set_xlabel("epoch")
fig.tight_layout()
fig.savefig(os.path.join(CONFIG["output_dir"], "plot_baseline.png"), dpi=130, bbox_inches="tight")
plt.show()

## So sánh với CAKD student

Tải `history_baseline.json` (tab Output) về máy, rồi chạy trong repo:

```bash
python tools/plot_report.py teacher_student.json history_baseline.json \
  --labels "CAKD student" "ResNet50 baseline" --outdir img/report
```

**Lưu ý khi đọc kết quả:** baseline này train "kiểu thường" (augment chuẩn, 30 epoch), còn run CAKD dùng
công thức nặng (ta_wide, mixup, EMA, 120 epoch). Nếu CAKD thắng, khoảng cách gồm cả distillation lẫn
công thức train — muốn quy công riêng cho teacher thì tăng `epochs` lên 120 cho cùng thời lượng học.